In [38]:
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import os
import transformers
import torch
from sklearn.metrics import accuracy_score, f1_score
# from sklearn.preprocessing import LabelEncoder

load_dotenv()

True

In [11]:
with open('qa_aggregated.json','r') as file:
    data=json.load(file)

In [12]:
for i in range(len(data)):
    data[i]['q_id']=i

In [13]:
len(data)

241

In [14]:
data[240]

{'topic': 'opportunities_future_responsiveness',
 'difficulty': 'hard',
 'question': 'What emerging opportunity for PEC hydrogels one can propose beyond traditional salt/pH/temperature responsiveness?',
 'options': {'A': 'Designing PEC hydrogels that cannot respond to any external stimulus to ensure stability',
  'B': 'Incorporating new response modes such as photocleavable groups for light-triggered degradation or cargo release',
  'C': 'Restricting PEC hydrogels to only bulk precipitate formation to improve processability',
  'D': 'Eliminating charged cargos because PEC domains cannot encapsulate macromolecules'},
 'correct_option': 'B',
 'answer': 'Incorporating new response modes such as photocleavable groups for light-triggered degradation or cargo release',
 'justification': {'anchor': 'Opportunities: imaginative response methods and photocleavable functionality',
  'rationale': 'The text suggests photocleavable groups (visible light) and other stimuli (mechanical force, biomolec

## Qwen3:8b

In [20]:
def prompt_template(question, options):
    return f"You are the chemistry expert. Please answer the question: {question}. \
           The answer options: {options}. As output, give only the symbol for the correct answer."

In [47]:
predictions=[]

for entry in data:
    prompt=prompt_template(entry['question'], entry['options'])
    response = chat(
      model='qwen3:8b',
      messages=[{'role': 'user', 'content': prompt }],
      think=True,
      stream=False,
    )
    predictions.append({'thinking':response.message.thinking, 'answer': response.message.content})

In [48]:
with open('qwen3_8b_ollama.json','w') as file:
    json.dump(predictions, file)

In [49]:
answers=[]
truth=[]

for i,entry in enumerate(predictions):
    answers.append(entry['answer'])
    truth.append(data[i]['correct_option'])

In [57]:
print('Acc: ', accuracy_score(answers,truth))
print('F1: ', f1_score(answers,truth,average='macro'))

Acc:  0.9543568464730291
F1:  0.7646631413175184


In [52]:
from collections import Counter

In [56]:
x=Counter(truth)
print(x)

Counter({'B': 69, 'C': 64, 'A': 55, 'D': 53})


In [65]:
for i, tup in enumerate(zip(answers,truth)):
    if(tup[0]!=tup[1]):
        print(i, 'prediction: ', tup[0], 'truth: ', tup[1])
        print(data[i]['question'])
        print(data[i]['options'])
        print("-----------------")
        print(predictions[i]['thinking'])
        print("*****************")

32 prediction:  D truth:  B
Which requirement is essential for facilitated transport carriers to function effectively inside a membrane?
{'A': 'Carriers must form irreversible bonds to prevent back-diffusion', 'B': 'Carriers must be well dispersed along diffusion pathways and remain unpoisoned/available for interaction', 'C': 'Carriers must be volatile so they can migrate to the feed interface', 'D': 'Carriers must be present only at the membrane surface, not in the bulk'}
-----------------
Okay, so I need to figure out which requirement is essential for facilitated transport carriers to function effectively inside a membrane. The options are A, B, C, D. Let me start by recalling what I know about facilitated transport.

Facilitated transport is a type of passive transport where molecules move across a membrane with the help of carrier proteins. Unlike simple diffusion, which doesn't require proteins, facilitated transport uses these carriers to help substances that can't easily pass t

# Deepseek-8b

In [68]:
predictions_d=[]

for entry in data:
    prompt=prompt_template(entry['question'], entry['options'])
    response = chat(
      model='deepseek-r1:8b',
      messages=[{'role': 'user', 'content': prompt }],
      think=True,
      stream=False,
    )
    predictions_d.append({'thinking':response.message.thinking, 'answer': response.message.content})

In [69]:
with open('deepseek_r1_8b_ollama.json','w') as file:
    json.dump(predictions_d, file)

In [70]:
answers=[]
truth=[]

for i,entry in enumerate(predictions_d):
    answers.append(entry['answer'])
    truth.append(data[i]['correct_option'])

In [71]:
print('Acc: ', accuracy_score(answers,truth))
print('F1: ', f1_score(answers,truth,average='macro'))

Acc:  0.9336099585062241
F1:  0.6246567230278505
